# 🕌 IlmGPT — Islamic Scholar Assistant
### RAG Pipeline using LangChain + ChromaDB + Google Gemini

**Built by:** Muhammad Wasif  
**Project:** IlmGPT — Ask questions about Quran & Hadith with authentic source citations

---

## 📋 Pipeline Overview
```
Step 1 → Download Dataset (Quran + Hadith CSV files)
Step 2 → Load & Prepare Documents
Step 3 → Chunk the Text
Step 4 → Create Embeddings (SentenceTransformer)
Step 5 → Store in ChromaDB (Vector Database)
Step 6 → Retrieve Relevant Chunks for a Query
Step 7 → Build RAG Prompt (English + Urdu)
Step 8 → Get Answer from Google Gemini
Step 9 → Show Answer with Source Citations
```

## 📂 Dataset Sources
- **Quran:** https://www.kaggle.com/datasets/zusmani/the-holy-quran  
  *(Download `en.csv` — English translation by Yusuf Ali)*
- **Hadith:** https://www.kaggle.com/datasets/fahd09/hadith-dataset  
  *(Download `hadiths.csv` — Bukhari, Muslim, Abu Dawud collections)*

**After downloading, place files in:**
```
ilmgpt/
  data/
    quran_en.csv
    hadiths.csv
```

---
## 🔑 Step 1: Set Your API Key

Get your FREE Gemini API key from: https://aistudio.google.com/app/apikey

**Free tier gives you:** 15 requests/minute, 1 million tokens/day — more than enough!

In [ ]:
import os

# ONLY LINE YOU NEED TO CHANGE — paste your Gemini API key here
os.environ["GOOGLE_API_KEY"] = "YOUR_GEMINI_API_KEY_HERE"

# Verify it is set
if os.environ.get("GOOGLE_API_KEY") == "YOUR_GEMINI_API_KEY_HERE":
    print("Please replace YOUR_GEMINI_API_KEY_HERE with your actual key")
else:
    print("Gemini API key is set!")

 Please replace YOUR_GEMINI_API_KEY_HERE with your actual key


---
## 📥 Step 2: Load Dataset

We load two CSV files:
- `quran_en.csv` → Surah number, Ayah number, English translation text
- `hadiths.csv`  → Collection name, Book, Hadith number, Hadith text

In [3]:
import pandas as pd
from pathlib import Path

# -- Load Quran CSV ---------------------------------------------------------
quran_path = Path("Data/Quran Multiple Language/en.yusufali.csv")
quran_df = pd.read_csv(quran_path)

print("=== Quran Dataset ===")
print(f"Path    : {quran_path}")
print(f"Shape   : {quran_df.shape}")
print(f"Columns : {list(quran_df.columns)}")
print(quran_df.head(3))

=== Quran Dataset ===
Path    : Data\Quran Multiple Language\en.yusufali.csv
Shape   : (6236, 3)
Columns : ['Surah', 'Ayah', 'Text']
   Surah  Ayah                                               Text
0      1     1  In the name of Allah, Most Gracious, Most Merc...
1      1     2  Praise be to Allah, the Cherisher and Sustaine...
2      1     3                      Most Gracious, Most Merciful;


In [4]:
# -- Load Hadith CSV --------------------------------------------------------
hadith_path = Path("Data/Hadees/all_hadiths_clean.csv")
hadith_df = pd.read_csv(hadith_path, encoding="latin1", low_memory=False)

print("=== Hadith Dataset ===")
print(f"Path    : {hadith_path}")
print(f"Shape   : {hadith_df.shape}")
print(f"Columns : {list(hadith_df.columns)}")
print(hadith_df.head(3))

=== Hadith Dataset ===
Path    : Data\Hadees\all_hadiths_clean.csv
Shape   : (34441, 9)
Columns : ['id', 'hadith_id', 'source', 'chapter_no', 'hadith_no', 'chapter', 'chain_indx', 'text_ar', 'text_en']
   id  hadith_id           source  chapter_no hadith_no  \
0   0          1   Sahih Bukhari            1        1    
1   1          2   Sahih Bukhari            1        2    
2   2          3   Sahih Bukhari            1        3    

                                   chapter  \
0  Revelation - ÙØªØ§Ø¨ Ø¨Ø¯Ø¡ Ø§ÙÙØ­Ù   
1  Revelation - ÙØªØ§Ø¨ Ø¨Ø¯Ø¡ Ø§ÙÙØ­Ù   
2  Revelation - ÙØªØ§Ø¨ Ø¨Ø¯Ø¡ Ø§ÙÙØ­Ù   

                              chain_indx  \
0   30418, 20005, 11062, 11213, 11042, 3   
1         30355, 20001, 11065, 10511, 53   
2  30399, 20023, 11207, 11013, 10511, 53   

                                             text_ar  \
0  Ø­Ø¯Ø«ÙØ§ Ø§ÙØ­ÙÙØ¯Ù Ø¹Ø¨Ø¯ Ø§ÙÙÙ Ø¨Ù...   
1  Ø­Ø¯Ø«ÙØ§ Ø¹Ø¨Ø¯ Ø§ÙÙÙ Ø¨Ù ÙÙØ³ÙØ ÙØ...   
2  Ø­Ø¯Ø«ÙØ§ ÙØ­ÙÙ Ø¨Ù Ø¨Ù

---
## 🧹 Step 3: Clean & Convert to LangChain Documents

Each Ayah and each Hadith becomes a LangChain `Document` object.
Metadata stores the source info for citations later.

In [5]:
from langchain_core.documents import Document

# -- Convert Quran rows -> Documents ----------------------------------------
quran_docs = []
for _, row in quran_df.iterrows():
    text = (
        row.get("Text")
        or row.get("text")
        or row.get("translation")
        or row.get("english")
        or str(row.iloc[-1])
    )

    surah = row.get("Surah") or row.get("surah") or row.get("sura") or row.get("chapter") or row.iloc[0]
    ayah = row.get("Ayah") or row.get("ayah") or row.get("verse") or row.get("aya") or row.iloc[1]

    if pd.isna(text) or str(text).strip() == "":
        continue

    doc = Document(
        page_content=str(text).strip(),
        metadata={
            "source": "Quran",
            "surah": int(surah) if str(surah).isdigit() else surah,
            "ayah": int(ayah) if str(ayah).isdigit() else ayah,
            "reference": f"Quran {surah}:{ayah}",
            "type": "quran",
        },
    )
    quran_docs.append(doc)

print(f"Quran Documents created: {len(quran_docs)}")
print("\nSample:")
print(f"  Text      : {quran_docs[0].page_content[:100]}...")
print(f"  Metadata  : {quran_docs[0].metadata}")

Quran Documents created: 6236

Sample:
  Text      : In the name of Allah, Most Gracious, Most Merciful....
  Metadata  : {'source': 'Quran', 'surah': 1, 'ayah': 1, 'reference': 'Quran 1:1', 'type': 'quran'}


In [6]:
# -- Convert Hadith rows -> Documents ---------------------------------------
hadith_docs = []
for _, row in hadith_df.iterrows():
    text = (
        row.get("text_en")
        or row.get("text")
        or row.get("hadith")
        or row.get("body")
        or row.get("content")
        or str(row.iloc[-1])
    )

    collection = row.get("source") or row.get("collection") or row.get("book_name") or "Unknown"
    book = row.get("chapter") or row.get("book") or ""
    number = row.get("hadith_no") or row.get("hadithNo") or row.get("number") or row.get("id") or ""

    if pd.isna(text) or str(text).strip() == "":
        continue

    doc = Document(
        page_content=str(text).strip(),
        metadata={
            "source": "Hadith",
            "collection": str(collection),
            "book": str(book),
            "number": str(number),
            "reference": f"{collection} - Hadith {number}",
            "type": "hadith",
        },
    )
    hadith_docs.append(doc)

print(f"Hadith Documents created: {len(hadith_docs)}")
print("\nSample:")
print(f"  Text      : {hadith_docs[0].page_content[:100]}...")
print(f"  Metadata  : {hadith_docs[0].metadata}")

Hadith Documents created: 33588

Sample:
  Text      : Narrated 'Umar bin Al-Khattab:                          I heard Allah's Apostle saying, "The reward ...
  Metadata  : {'source': 'Hadith', 'collection': ' Sahih Bukhari ', 'book': 'Revelation - Ù\x83ØªØ§Ø¨ Ø¨Ø¯Ø¡ Ø§Ù\x84Ù\x88Ø\xadÙ\x89', 'number': ' 1 ', 'reference': ' Sahih Bukhari  - Hadith  1 ', 'type': 'hadith'}


In [7]:
# ── Combine all documents ───────────────────────────────────────────────────
all_documents = quran_docs + hadith_docs

print(f"Total Documents : {len(all_documents)}")
print(f"  Quran Ayahs   : {len(quran_docs)}")
print(f"  Hadiths       : {len(hadith_docs)}")

Total Documents : 39824
  Quran Ayahs   : 6236
  Hadiths       : 33588


---
## ✂️ Step 4: Text Chunking

Ayahs and Hadiths are already short — but some Hadiths are very long.
We split anything over 500 characters into smaller chunks.

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Smaller chunk size because Quran ayahs are naturally short
# Overlap preserves context across long Hadiths
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size    = 500,
    chunk_overlap = 100,
    separators    = ["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(all_documents)

print(f"Documents before chunking : {len(all_documents)}")
print(f"Chunks after splitting     : {len(chunks)}")
print(f"\nSample chunk:")
print(f"  Content  : {chunks[0].page_content[:150]}")
print(f"  Metadata : {chunks[0].metadata}")

Documents before chunking : 39824
Chunks after splitting     : 55818

Sample chunk:
  Content  : In the name of Allah, Most Gracious, Most Merciful.
  Metadata : {'source': 'Quran', 'surah': 1, 'ayah': 1, 'reference': 'Quran 1:1', 'type': 'quran'}


---
## 🔢 Step 5: Create Embeddings

Converting every chunk into a 384-dimension vector using SentenceTransformer.
This step may take a few minutes for large datasets.

In [9]:
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import numpy as np

print("Loading embedding model...")
# all-MiniLM-L6-v2 → fast, lightweight, 384 dimensions, works great for semantic search
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print(" Model loaded!")

# Extract raw text from chunks
texts     = [chunk.page_content for chunk in chunks]
metadatas = [chunk.metadata     for chunk in chunks]
ids       = [str(i)             for i in range(len(chunks))]

print(f"\nEmbedding {len(texts)} chunks... (this may take 2-5 minutes)")

# Encode in batches for memory efficiency
embeddings = embedding_model.encode(
    texts,
    batch_size        = 64,
    show_progress_bar = True
)

print(f"\n Embeddings created!")
print(f"Shape: {embeddings.shape}  →  ({len(texts)} chunks × 384 dimensions)")

d:\Quran-Hadees-RAG_Langchain_Project\.venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


Loading embedding model...
 Model loaded!

Embedding 55818 chunks... (this may take 2-5 minutes)


Batches: 100%|██████████| 873/873 [26:03<00:00,  1.79s/it]



 Embeddings created!
Shape: (55818, 384)  →  (55818 chunks × 384 dimensions)


---
## 🗄️ Step 6: Store in ChromaDB

Saving everything to disk so we never need to re-embed again.

In [10]:
import chromadb

# Connect to persistent ChromaDB (saves to db/chroma folder)
chroma_client = chromadb.PersistentClient(path="db/chroma")

# Create collection — delete old one if re-running this cell
try:
    chroma_client.delete_collection("ilmgpt_collection")
    print("Old collection deleted — creating fresh one")
except:
    pass

collection = chroma_client.create_collection(
    name     = "ilmgpt_collection",
    metadata = {"hnsw:space": "cosine"}  # Cosine similarity for text
)

# Store in batches of 500 (ChromaDB limit per call)
BATCH_SIZE = 500
total      = len(texts)

for start in range(0, total, BATCH_SIZE):
    end = min(start + BATCH_SIZE, total)
    collection.add(
        documents  = texts[start:end],
        embeddings = embeddings[start:end].tolist(),
        metadatas  = metadatas[start:end],
        ids        = ids[start:end]
    )
    print(f"  Stored chunks {start} → {end}")

print(f"\n All {collection.count()} chunks stored in ChromaDB!")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


  Stored chunks 0 → 500
  Stored chunks 500 → 1000
  Stored chunks 1000 → 1500
  Stored chunks 1500 → 2000
  Stored chunks 2000 → 2500
  Stored chunks 2500 → 3000
  Stored chunks 3000 → 3500
  Stored chunks 3500 → 4000
  Stored chunks 4000 → 4500
  Stored chunks 4500 → 5000
  Stored chunks 5000 → 5500
  Stored chunks 5500 → 6000
  Stored chunks 6000 → 6500
  Stored chunks 6500 → 7000
  Stored chunks 7000 → 7500
  Stored chunks 7500 → 8000
  Stored chunks 8000 → 8500
  Stored chunks 8500 → 9000
  Stored chunks 9000 → 9500
  Stored chunks 9500 → 10000
  Stored chunks 10000 → 10500
  Stored chunks 10500 → 11000
  Stored chunks 11000 → 11500
  Stored chunks 11500 → 12000
  Stored chunks 12000 → 12500
  Stored chunks 12500 → 13000
  Stored chunks 13000 → 13500
  Stored chunks 13500 → 14000
  Stored chunks 14000 → 14500
  Stored chunks 14500 → 15000
  Stored chunks 15000 → 15500
  Stored chunks 15500 → 16000
  Stored chunks 16000 → 16500
  Stored chunks 16500 → 17000
  Stored chunks 17000 → 

---
## 🔁 Step 6b: Load Existing DB (Skip Embedding Next Time)

After running Step 5 & 6 once, just run this cell on future sessions.

In [11]:
import chromadb
from sentence_transformers import SentenceTransformer

# Reload model and database
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
chroma_client   = chromadb.PersistentClient(path="db/chroma")
collection      = chroma_client.get_collection("ilmgpt_collection")

print(f" Reconnected! Database has {collection.count()} chunks ready")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


 Reconnected! Database has 55818 chunks ready


---
## 🔎 Step 7: Retrieval — Find Relevant Chunks

In [12]:
def retrieve(query: str, top_k: int = 5, source_filter: str = None) -> dict:
    """
    Retrieve top-k relevant chunks from ChromaDB for a given query.

    Args:
        query         : User's question in English or Urdu
        top_k         : Number of chunks to retrieve
        source_filter : 'quran' or 'hadith' to filter by source, None for both

    Returns:
        Dictionary with documents and metadatas
    """
    # Convert query to embedding vector
    query_vector = embedding_model.encode([query]).tolist()

    # Optional: filter by source type (quran or hadith)
    where_filter = {"type": source_filter} if source_filter else None

    results = collection.query(
        query_embeddings = query_vector,
        n_results        = top_k,
        where            = where_filter
    )

    return results


# ── Test retrieval ──────────────────────────────────────────────────────────
test_query   = "What does Islam say about patience?"
test_results = retrieve(test_query, top_k=3)

print(f"Query: '{test_query}'\n")
for i, (doc, meta) in enumerate(zip(
    test_results["documents"][0],
    test_results["metadatas"][0]
)):
    print(f"--- Result {i+1} [{meta['reference']}] ---")
    print(doc[:200])
    print()

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Query: 'What does Islam say about patience?'

--- Result 1 [ Jami' al-Tirmidhi  - Hadith  3442 ] ---
can you have patience about a thing which you know not? He said: If Allah wills, you will find me patient, and I will not disobey you at all (18:67-69)

--- Result 2 [ Jami' al-Tirmidhi  - Hadith  2156 ] ---
. And whoever remains patient, Allah will make him patient. Nobody can be given a blessing better and more encompassing than patience."

--- Result 3 [ Sunan Ibn Majah  - Hadith  4168 ] ---
It was narrated from Ibn âUmar that the Messenger of Allah (ï·º)said:                      âThe believer who mixes with people and bears their annoyancewith patience will have a greater reward tha



---
## 📝 Step 8: Build Bilingual RAG Prompt (English + Urdu)

In [13]:
def build_prompt(query: str, results: dict) -> str:
    """
    Build a bilingual RAG prompt for Google Gemini.
    Instructs the model to answer in both English and Urdu
    using only the provided Islamic context.

    Args:
        query   : User's question
        results : Retrieved chunks from ChromaDB

    Returns:
        Complete prompt string
    """
    # Build numbered context with source references
    context_parts = []
    for i, (doc, meta) in enumerate(zip(
        results["documents"][0],
        results["metadatas"][0]
    )):
        ref = meta.get("reference", "Unknown Source")
        context_parts.append(f"[{i+1}] {ref}:\n{doc}")

    context = "\n\n".join(context_parts)

    prompt = f"""You are IlmGPT, a respectful and knowledgeable Islamic Scholar Assistant.
Your role is to answer questions about Islam using ONLY the Quran and Hadith sources provided below.

STRICT RULES:
1. Answer ONLY from the provided context. Do not use outside knowledge.
2. If the answer is not in the context, say: "This specific matter is not covered in the provided sources. Please consult a qualified scholar."
3. Always cite the source reference number [1], [2] etc. in your answer.
4. Be respectful, humble, and scholarly in tone.
5. Provide answer in TWO parts: first in English, then in Urdu.

--- ISLAMIC SOURCES ---
{context}
--- END OF SOURCES ---

Question: {query}

Please answer in this exact format:

**English Answer:**
[Your detailed answer in English with citations like [1], [2]]

**اردو جواب:**
[Same answer translated in clear Urdu]

**Sources Used:**
[List the references you cited]"""

    return prompt


# Test prompt building
sample_prompt = build_prompt(test_query, test_results)
print(f"Prompt length: {len(sample_prompt)} characters")
print("\n--- Prompt Preview ---")
print(sample_prompt[:500] + "...")

Prompt length: 1612 characters

--- Prompt Preview ---
You are IlmGPT, a respectful and knowledgeable Islamic Scholar Assistant.
Your role is to answer questions about Islam using ONLY the Quran and Hadith sources provided below.

STRICT RULES:
1. Answer ONLY from the provided context. Do not use outside knowledge.
2. If the answer is not in the context, say: "This specific matter is not covered in the provided sources. Please consult a qualified scholar."
3. Always cite the source reference number [1], [2] etc. in your answer.
4. Be respectful, hum...


---
## 🤖 Step 9: Get Answer from Google Gemini

In [19]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

# Initialize Gemini Flash model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0.1,
    max_tokens=1024,
    google_api_key=os.environ.get("GOOGLE_API_KEY"),
)

print("Gemini LLM initialized!")

Gemini LLM initialized!


In [20]:
def ask_ilmgpt(question: str, top_k: int = 5, source_filter: str = None) -> str:
    """
    Complete IlmGPT pipeline: question → retrieve → prompt → Gemini → answer

    Args:
        question      : Your Islamic question (English or Urdu)
        top_k         : How many source chunks to retrieve (default: 5)
        source_filter : 'quran', 'hadith', or None for both

    Returns:
        Bilingual answer string with citations
    """
    print(f"🔍 Searching Islamic sources for: '{question}'")

    # Step 1: Retrieve relevant chunks
    results = retrieve(question, top_k=top_k, source_filter=source_filter)
    print(f" Found {len(results['documents'][0])} relevant chunks")

    # Step 2: Build RAG prompt
    prompt = build_prompt(question, results)

    # Step 3: Send to Gemini
    print(" Asking Gemini...")
    response = llm.invoke([HumanMessage(content=prompt)])
    answer   = response.content

    # Step 4: Show citations
    print("\n" + "="*60)
    print(answer)
    print("="*60)

    return answer


# ── Test the full pipeline ──────────────────────────────────────────────────
answer = ask_ilmgpt("What does Islam say about the importance of prayer?")

🔍 Searching Islamic sources for: 'What does Islam say about the importance of prayer?'
 Found 5 relevant chunks
 Asking Gemini...


Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
Please retry in 25.07173219s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googlea

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
Please retry in 22.447819222s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quo

In [ ]:
# ── More test questions ─────────────────────────────────────────────────────

# Filter only Quran
ask_ilmgpt("What does the Quran say about forgiveness?", source_filter="quran")

In [ ]:
# Filter only Hadith
ask_ilmgpt("What did the Prophet say about treating neighbors?", source_filter="hadith")

In [ ]:
# Urdu question works too!
ask_ilmgpt("نماز کی اہمیت کیا ہے؟")

---
##  Pipeline Complete!

Your IlmGPT notebook is fully working. Now run the Streamlit app:

```bash
streamlit run app/streamlit_app.py
```

---
## 📊 What This Project Demonstrates (for CV/LinkedIn)

| Skill | Technology Used |
|---|---|
| RAG Pipeline | LangChain |
| Vector Database | ChromaDB |
| Semantic Embeddings | SentenceTransformers |
| LLM Integration | Google Gemini 1.5 Flash |
| Data Processing | Pandas |
| Web Application | Streamlit |
| Bilingual NLP | English + Urdu |
| Dataset | Quran + Hadith (10,000+ texts) |

In [ ]:
import google.generativeai as genai

genai.configure(api_key=os.environ.get("GOOGLE_API_KEY"))

supported = []
for m in genai.list_models():
    methods = getattr(m, "supported_generation_methods", [])
    if "generateContent" in methods:
        supported.append(m.name)

print("Supported models for generateContent:")
for name in supported[:30]:
    print(name)

Supported models for generateContent:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
